In [1]:
import numpy as np

In [ ]:
def initialize_parameters(layer_dims):
    L = len(layer_dims)
    #L denotes number of layers including input and output
    parameters = {}
    for i in range(1,L):
        parameters['W'+str(i)] = np.random.randn(layer_dims[i],layer_dims[i-1])
        parameters['b'+str(i)] = np.zeros((layer_dims[i],1))

    return parameters
def initialize_parameters_he(layer_dims):
    L = len(layer_dims)
    parameters = {}

    for i in range(1, L):
        parameters['W'+str(i)] = (
            np.random.randn(layer_dims[i], layer_dims[i-1])
            * np.sqrt(2 / layer_dims[i-1])
        )

        parameters['b'+str(i)] = np.zeros((layer_dims[i],1))

    return parameters

## Forward Propagation

In [3]:
def forward_step(A,W,b):
    Z = np.dot(W,A) + b
    return Z,(A,W,b)

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True),Z

def relu(Z):
    return np.maximum(0, Z), Z

def activation_forward_step(A_prev,W,b,activation):
    Z, linear_cache = forward_step(A_prev,W,b)
    if activation == 'relu':
        A,activation_cache = relu(Z)
    elif activation == 'softmax':
        A,activation_cache = softmax(Z)
    return A, (linear_cache,activation_cache)

def model_forward(parameters,X):
    L = len(parameters)//2
    caches=[]
    A_prev = X
    for i in range(1,L):
        A,cache = activation_forward_step(A_prev,parameters['W'+str(i)],parameters['b'+str(i)],activation = 'relu')
        caches.append(cache)
        A_prev = A
    AL,cache = activation_forward_step(A, parameters['W'+str(L)], parameters['b'+str(L)],activation = 'softmax')
    caches.append(cache)
    return AL,caches

## Backward Propagation

In [4]:
def compute_cost(Y,AL):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(AL + 1e-8)) / m
    return cost

def relu_backward(dA, activation_cache):
    Z = activation_cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ



### Linear Backward

For layer $l$, the linear part is: $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ (followed by an activation).

Suppose you have already calculated the derivative $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$. You want to get $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$.



The three outputs $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ are computed using the input $dZ^{[l]}$.

Here are the formulas you need:
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]}$$


$A^{[l-1] T}$ is the transpose of $A^{[l-1]}$.

With the help of above defined functions, we can now use them to further evaluate dA_prev, dW, db


In [5]:
#Above notation has been picked up from andrew ng's deep learning course module
def backward_step(dZ,cache):
    A_prev,W,b = cache
    m=A_prev.shape[1]
    dW = 1/m * np.dot(dZ,A_prev.T)
    db = 1/m * np.sum(dZ, axis = 1, keepdims =True)
    dA_prev = np.dot(W.T, dZ)
    return dA_prev, dW, db
#Now, depending on the choice of activation function, the gradient will also vary
def activation_backward_step(dA, cache, activation):
    linear_cache, activation_cache = cache

    if activation == 'relu':
        dZ = relu_backward(dA, activation_cache)
        dA_prev, dW, db = backward_step(dZ, linear_cache)

        return dA_prev, dW, db

Now, we will evaluate the gradients and update parameters.

In [6]:
def model_backward(AL, Y, caches):
    grads = {}
    L = len(caches)
    m = AL.shape[1]
    dZL = AL - Y
    current_cache = caches[L - 1]
    grads["dA" + str(L-1)], grads["dW" + str(L)], grads["db" + str(L)] = backward_step(dZL, current_cache[0])
    for l in reversed(range(L - 1)):
        current_cache = caches[l]
        dA_prev, dW, db = activation_backward_step(grads["dA" + str(l+1)],current_cache,"relu")
        grads["dA" + str(l)] = dA_prev
        grads["dW" + str(l + 1)] = dW
        grads["db" + str(l + 1)] = db

    return grads

def update_parameters(parameters,learning_rate,grads):
    L = len(parameters)//2
    for i in range(1,L+1):
        parameters['W'+str(i)] -= learning_rate*grads['dW'+str(i)]
        parameters["b" + str(i)] -= learning_rate * grads["db" + str(i)]

    return parameters

Now, to combine it all, we define the function to obtain final parameters

In [ ]:
import time
import psutil
import os
def train_parameters(X,Y,learning_rate,num_iterations,layer_dims,print_cost=True):
    process = psutil.Process(os.getpid())
    parameters = initialize_parameters_he(layer_dims)
    costs = []
    memory_usage = []
    peak_memory = 0
    tic = time.time()
    for iter in range(num_iterations):
        AL,caches = model_forward(parameters,X)
        grads = model_backward(AL,Y,caches)
        parameters = update_parameters(parameters,learning_rate,grads)
        cost = compute_cost(Y,AL)
        current_memory = process.memory_info().rss / (1024 ** 2)
        memory_usage.append(current_memory)
        peak_memory = max(peak_memory, current_memory)
        if print_cost and iter%100==0:
            print("Cost after iteration %d: %f" % (iter, cost))
            print("Memory Usage: %.2f MB" % current_memory)
        costs.append(cost)
    toc = time.time()

    print("\nPeak Memory Usage During Training: %.2f MB" % peak_memory)

    return parameters, costs, memory_usage, peak_memory, toc-tic

def predict(X,parameters):
    AL,caches = model_forward(parameters,X)
    predictions = np.argmax(AL,axis=0)
    return predictions

Our model is now ready to be implemented


In [42]:
from google.colab import files
uploaded = files.upload()



In [8]:
import pandas as pd
df_train = pd.read_csv('mnist_train.csv')
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to 28x28
dtypes: int64(785)
memory usage: 359.3 MB


In [44]:
df_train.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
X = df_train.drop('label', axis=1).values
Y = df_train['label'].values


We need to correct the shape of Y to the shape(10,m) where m is the number of training examples here


In [ ]:
X = X.T
def one_hot(Y, num_classes=10):
    return np.eye(num_classes)[Y.reshape(-1)].T

In [11]:
Y = one_hot(Y)
print(Y.shape,X.shape)

(10, 60000) (784, 60000)


In [ ]:
def accuracy(predictions,Y_true):
    acc = np.mean(predictions == Y_true) * 100
    return acc
def memory(parameters,grads):
    total_bytes = (sum(v.nbytes for v in parameters.values()) +sum(v.nbytes for v in grads.values()))
    return total_bytes


In [53]:
type(X)

numpy.ndarray

Now, we begin training of the neural network.

In [13]:
layer_dims = [784,128,64,10]
X=X/255.0
X = X.astype(np.float32)
X_train_smaller = X[:, :20000]
Y_train_smaller = Y[:, :20000]
learning_rate = 0.03
num_iterations=3000
parameters,costs,memory_usage, peak_memory,training_time_scratch = train_parameters(X_train_smaller,Y_train_smaller,learning_rate,num_iterations,layer_dims,print_cost=True)

Cost after iteration 0: 2.335330
Memory Usage: 1197.03 MB
Cost after iteration 100: 0.687915
Memory Usage: 1197.03 MB
Cost after iteration 200: 0.455640
Memory Usage: 1197.04 MB
Cost after iteration 300: 0.379169
Memory Usage: 1197.06 MB
Cost after iteration 400: 0.338620
Memory Usage: 1197.06 MB
Cost after iteration 500: 0.311800
Memory Usage: 1197.07 MB
Cost after iteration 600: 0.291654
Memory Usage: 1197.08 MB
Cost after iteration 700: 0.275378
Memory Usage: 1197.09 MB
Cost after iteration 800: 0.261631
Memory Usage: 1197.09 MB
Cost after iteration 900: 0.249642
Memory Usage: 1197.11 MB
Cost after iteration 1000: 0.238909
Memory Usage: 1197.11 MB
Cost after iteration 1100: 0.229197
Memory Usage: 1197.13 MB
Cost after iteration 1200: 0.220336
Memory Usage: 1197.13 MB
Cost after iteration 1300: 0.212237
Memory Usage: 1197.16 MB
Cost after iteration 1400: 0.204760
Memory Usage: 1197.16 MB
Cost after iteration 1500: 0.197796
Memory Usage: 1197.17 MB
Cost after iteration 1600: 0.191278


In [14]:
import pandas as pd
df_test = pd.read_csv('mnist_test.csv')
X_test = df_test.drop('label', axis=1).values
Y_test = df_test['label'].values
Y_test = one_hot(Y_test,num_classes=10)
X_test = X_test.T
print(Y_test.shape,X_test.shape)
predictions = predict(X_test,parameters)
Y_true_test = np.argmax(Y_test,axis=0)
acc = accuracy(predictions,Y_true_test)
print(f"Test Accuracy: {acc:.2f}%")
print(f"Training time: {training_time_scratch:.2f}s")



(10, 10000) (784, 10000)
Test Accuracy: 94.60%
Training time: 2288.18s


In [ ]:
!cat /proc/cpuinfo


processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2200.238
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4400.47
clflush size	: 64
cache_alignment	: 64
address sizes

In [ ]:
import os
print(os.cpu_count())

2


In [15]:
print(peak_memory)

1197.296875


In [16]:
def confusion_matrix(y_true, y_pred, num_classes=10):
    cm = np.zeros((num_classes, num_classes), dtype=int)

    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1

    return cm
cm = confusion_matrix(Y_true_test,predictions)
print(cm)

[[ 966    0    0    4    0    3    5    1    1    0]
 [   0 1114    4    2    0    1    3    1   10    0]
 [   9    1  978   10    6    0    7   10    9    2]
 [   2    0   12  953    0   10    1    6   19    7]
 [   1    1    3    1  914    0   12    4   11   35]
 [  11    1    1   31    8  795   12    2   24    7]
 [  10    3    3    1   10    6  920    0    5    0]
 [   3   10   22    9    3    1    0  951    1   28]
 [   4    2    7   13    6    3    5    7  925    2]
 [   8    7    1   12   20    0    1   10    6  944]]
